# 実習② 手の動きの質を見る

SAM-Body4Dが動画から取り出した手首の位置を使って、手の速さの変化（速度曲線）をグラフにし、動きの滑らかさを数にします。
動画は、高齢の女性がテーブルでグラスに水を注ぎ、手に取る8秒の場面です。

## このノートの使い方（実習①と同じ）

- セルの左の▶を押すと、そのセルが実行され、すぐ下に結果が出ます（Shift＋Enterでも同じ）。
- 上のセルから順に、全部押してください。下のセルは、上のセルで作ったものを使います。
- どこかを書き換えたら、そのセルから下を全部押し直してください。
- 結果が出ない・赤い文字が出たときは、メニューの［ランタイム］→［すべてのセルを実行］を押せば、上から順に押し直せます。

## 1. 道具を読み込む

実習①と同じく、`import`で道具箱を開けます。今回は、曲線をなめらかにする道具（`savgol_filter`）も使います。

In [ ]:
!pip install -q matplotlib-fontja
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib_fontja
from scipy.signal import savgol_filter, find_peaks
print('道具の準備ができました')

## 2. データを持ってくる

配られたIDを`'`と`'`の間に貼り付けてから▶を押してください（実習①とは別のIDです）。

In [ ]:
FILE_ID = 'YOUR_FILE_ID_HERE'
!gdown {FILE_ID} -O reaching_output.json

## 3. データを開いて大きさを見る

実習①と同じ形です。ファイルを開いて中身を`data`にしまい、22点の位置（`joints`）と1秒あたりのコマ数（`fps`）を取り出します。

In [ ]:
with open('reaching_output.json') as f:
    data = json.load(f)
joints = np.array(data['joints'])
fps = data['meta']['fps']
n_frames = len(joints)
print('コマ数', n_frames)
print('動画の長さ（秒）', n_frames / fps)

## 4. 手首の位置を取り出す

`joints[:, 21]`は「全部のコマの、21番の点（右手首）」です。`:`は「全部」という意味です。
`side`は実習①と同じく、右か左かを入れておく変数です。

In [ ]:
side = 'right'
if side == 'right':
    wrist = joints[:, 21]
else:
    wrist = joints[:, 20]
print('手首の位置の数', len(wrist))

## 5. 手の速さを出す

速さ＝1コマで動いた距離 × 1秒あたりのコマ数、です。
`np.diff`で隣どうしのコマの差（動いた向きと量）を出し、`np.linalg.norm`でその長さ（動いた距離、メートル）にします。

In [ ]:
step = np.linalg.norm(np.diff(wrist, axis=0), axis=1)
speed = step * fps
time = np.arange(len(speed)) / fps
plt.plot(time, speed)
plt.xlabel('時間 [秒]')
plt.ylabel('手首の速さ [m/s]')
plt.show()

## 6. 曲線をなめらかにする

生の速さは、コマごとにぎざぎざです。推定した位置が、コマごとに少しずつ揺れるためです。
`savgol_filter`で前後15コマを使ってなめらかにし、`np.clip`で0未満になったところを0にそろえます。

In [ ]:
speed_smooth = savgol_filter(speed, window_length=15, polyorder=2)
speed_smooth = np.clip(speed_smooth, 0, None)
plt.plot(time, speed, color='lightgray', label='生')
plt.plot(time, speed_smooth, color='crimson', label='なめらか')
plt.xlabel('時間 [秒]')
plt.ylabel('手首の速さ [m/s]')
plt.legend()
plt.show()

## 7. 速さの山を数える

実習①の挙上のピークと同じ道具（`find_peaks`）で、速さの山を数えます。いちばん速い山の10%より低い山は数えません。
健常な人の1回のリーチなら、山は1つ（ベル型）です。

In [ ]:
top = speed_smooth.max()
peaks, _ = find_peaks(speed_smooth, height=top * 0.1,
                      prominence=top * 0.1)
print('いちばん速いとき（m/s）', round(top, 3))
print('山の数', len(peaks))

## 8. 軌跡の直線性を数にする

直線性SI＝始点から終点までの直線距離 ÷ 実際に通った道のりの長さ、です。1に近いほどまっすぐです。

In [ ]:
path = step.sum()
straight = np.linalg.norm(wrist[-1] - wrist[0])
print('道のり（m）', round(path, 3))
print('直線距離（m）', round(straight, 3))
print('直線性SI', round(straight / path, 3))

## 9. 右と左を比べる

ここまでの計算を、右と左の両方でまとめて行います。このセルの関数は、4〜8のセルを1つにまとめたものです。中身は読み飛ばしてかまいません。
役割：手の動きの指標をまとめて出す。入力：22点の時系列、手首の番号、1秒あたりのコマ数。出力：指標の一覧。

左右差の指数 LSI＝評価したい側 ÷ 反対側 × 100（%）です。

In [ ]:
def reach_metrics(joints, wrist_idx, fps):
    w = joints[:, wrist_idx]
    st = np.linalg.norm(np.diff(w, axis=0), axis=1)
    sp = savgol_filter(st * fps, window_length=15, polyorder=2)
    sp = np.clip(sp, 0, None)
    pk, _ = find_peaks(sp, height=sp.max() * 0.1,
                       prominence=sp.max() * 0.1)
    si = np.linalg.norm(w[-1] - w[0]) / st.sum()
    return {'いちばん速いとき': round(sp.max(), 3),
            '山の数': len(pk),
            '直線性SI': round(si, 3)}

right = reach_metrics(joints, 21, fps)
left = reach_metrics(joints, 20, fps)
for name in right:
    lsi = right[name] / left[name] * 100
    print(name, '右', right[name], '左', left[name],
          'LSI', round(lsi, 1))

## 10. 実習の問い

グラフと数を見ながら、気づいたことをメモしてください。

1. なめらかにした速度曲線は、教科書のベル型（山が1つ）に見えるか。山はいくつあったか。
2. この8秒には、水を注ぐ・グラスを手に取る、の両方が入っている。山の数や直線性SIは「1回のリーチ」の値と言えるか。
3. 1回のリーチの値として測りたいなら、撮るとき・数えるときに何を決めておけばよいか。
4. 右と左の差（LSI）を、患側と健側の比較に使うとしたら、何をそろえて撮る必要があるか。